# 03 — Train the PyTorch and TensorFlow neural baselines

Here I train small neural networks with PyTorch and TensorFlow/Keras using the same data partitions. The source features are loaded with `scaled=False`. The workflow separates model fitting, early-stopping validation, conformal calibration and final testing. The feature scaler is fitted only on the model-fitting subset, then applied unchanged to the other subsets. Calibration observations are not used to train the models or choose the best epoch.

This is a small research benchmark, not a clinical prediction system.

## Install the optional dependencies

From the repository root, install the project with its neural-network dependencies:

```bash
pip install -e ".[neural]"
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from diabetes_ml.data import create_experiment_splits
from diabetes_ml.evaluation.metrics import calculate_regression_metrics
from diabetes_ml.neural.neural_data import prepare_neural_datasets
from diabetes_ml.neural.pytorch_model import PyTorchTrainer
from diabetes_ml.neural.tensorflow_model import TensorFlowTrainer

splits = create_experiment_splits(test_size=0.20, random_state=42)
data = prepare_neural_datasets(
    splits.X_train,
    splits.y_train,
    splits.X_test,
    splits.y_test,
    val_fraction=0.20,
    cal_fraction=0.20,
    random_state=42,
)

print(f"Model-fitting rows: {len(data.X_train)}")
print(f"Early-stopping validation rows: {len(data.X_val)}")
print(f"Conformal calibration rows (not used below): {len(data.X_cal)}")
print(f"Held-out test rows: {len(data.X_test)}")

## Train the PyTorch model

The trainer monitors validation MSE and restores the weights from the best validation epoch. The held-out test set is not used during training or early stopping.

In [ ]:
pytorch = PyTorchTrainer(
    in_features=data.X_train.shape[1],
    epochs=200,
    patience=20,
    batch_size=32,
    random_state=42,
)
pytorch.fit(data.X_train, data.y_train, data.X_val, data.y_val)
pytorch_predictions = pytorch.predict(data.X_test)
pytorch_metrics = calculate_regression_metrics(data.y_test.ravel(), pytorch_predictions)
print("PyTorch metrics:", pytorch_metrics)
print("PyTorch best validation epoch:", min(pytorch.history, key=lambda row: row["val_mse"])["epoch"])

## Train the TensorFlow/Keras model

The TensorFlow/Keras model uses the same fitting and validation partitions as the PyTorch model. It also restores its weights from the best validation epoch before the test metrics are calculated.

In [ ]:
tensorflow_model = TensorFlowTrainer(
    in_features=data.X_train.shape[1],
    epochs=200,
    patience=20,
    batch_size=32,
    random_state=42,
)
tensorflow_model.fit(data.X_train, data.y_train, data.X_val, data.y_val)
tensorflow_predictions = tensorflow_model.predict(data.X_test)
tensorflow_metrics = calculate_regression_metrics(data.y_test.ravel(), tensorflow_predictions)
print("TensorFlow metrics:", tensorflow_metrics)
print("TensorFlow best validation epoch:", min(tensorflow_model.history_records, key=lambda row: row["val_mse"])["epoch"])

In [ ]:
comparison = pd.DataFrame([
    {"framework": "PyTorch", **pytorch_metrics},
    {"framework": "TensorFlow/Keras", **tensorflow_metrics},
])
display(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, history, label in [
    (axes[0], pytorch.history, "PyTorch"),
    (axes[1], tensorflow_model.history_records, "TensorFlow/Keras"),
]:
    axis.plot([row["epoch"] for row in history], [row["train_mse"] for row in history], label="Training MSE")
    axis.plot([row["epoch"] for row in history], [row["val_mse"] for row in history], label="Validation MSE")
    axis.set_title(label)
    axis.set_xlabel("Epoch")
    axis.set_ylabel("MSE")
    axis.legend()
fig.tight_layout()
plt.show()

## Interpreting the results

Look at the training and validation curves together to consider whether the model is underfitting or overfitting. MAE and RMSE are measured in the target's units, and R² can be negative on held-out data. The results depend on one split, one small network architecture and the training settings used here; they do not show that one framework is generally better.

This notebook reports point-prediction metrics only. It does not fit or assess conformal intervals, and the reserved calibration subset is intentionally left unused. The benchmark has not been clinically validated and should not be used to make decisions about individual patients.